# Notebook 8: Hierarchical Sliding Window Inference (Map-Reduce) vs Qwen Long-Context

Thử nghiệm Kỹ thuật Sliding Window để giải quyết vấn đề Truncation (Cắt cụt) của các bài báo dài trên mô hình BARTpho (max_length=1024).
Để tăng tính mạch lạc (Coherence), ta sử dụng cơ chế Hierarchical (Map-Reduce):
1. Map: Chia bài báo thành các chunks, tóm tắt từng chunk.
2. Reduce: Nối các bản tóm tắt lại, và cho mô hình tóm tắt một lần cuối cùng.

So sánh với: Khả năng đọc nguyên khối (Native Long-Context) của Qwen2.5-0.5B (max_length=2048)

In [ ]:
!pip install -q transformers datasets evaluate rouge_score bert_score sacrebleu accelerate peft "torchao>=0.16.0"

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
from peft import PeftModel
import evaluate
import numpy as np
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Load Data
Chỉ test trên các bài báo DÀI (ví dụ: > 800 từ) trong tập test.

In [ ]:
test_df = pd.read_csv("test_1k.csv")
long_test_df = test_df[test_df['article'].apply(lambda x: len(str(x).split())) > 800].reset_index(drop=True)
print(f"Số lượng bài báo dài (>800 từ) trong tập test: {len(long_test_df)}")

# Lấy 50 bài dài nhất để tiết kiệm thời gian chạy (vì chạy sliding window + qwen tốn nhiều thời gian)
# long_test_df = long_test_df.head(50) # B? comment n?u ch? mu?n test th?
articles = long_test_df["article"].tolist()
references = long_test_df["abstract"].tolist()

## 2. BARTpho: Truncation & Hierarchical Sliding Window

In [ ]:
BARTPHO_DIR = "./bartpho_full_ft_final"
normal_preds = []
hierarchical_preds = []

if os.path.exists(BARTPHO_DIR):
    tokenizer_b = AutoTokenizer.from_pretrained(BARTPHO_DIR)
    model_b = AutoModelForSeq2SeqLM.from_pretrained(BARTPHO_DIR).to(device)
    model_b.eval()
    
    def normal_summarize(text):
        inputs = tokenizer_b(text, return_tensors="pt", max_length=1024, truncation=True, padding="max_length").to(device)
        with torch.no_grad():
            outputs = model_b.generate(**inputs, max_length=150, num_beams=4, length_penalty=2.0)
        return tokenizer_b.decode(outputs[0], skip_special_tokens=True)

    def hierarchical_sliding_summarize(text, chunk_size=512, overlap=50):
        tokens = tokenizer_b(text, return_tensors="pt", truncation=False)["input_ids"][0]
        chunks = []
        start = 0
        while start < len(tokens):
            end = min(start + chunk_size, len(tokens))
            chunks.append(tokens[start:end])
            if end == len(tokens): break
            start += (chunk_size - overlap)
            
        chunk_summaries = []
        for chunk_tokens in chunks:
            chunk_text = tokenizer_b.decode(chunk_tokens, skip_special_tokens=True)
            inputs = tokenizer_b(chunk_text, return_tensors="pt", max_length=1024, truncation=True).to(device)
            with torch.no_grad():
                outputs = model_b.generate(inputs["input_ids"], attention_mask=inputs["attention_mask"], max_length=64, num_beams=4)
            chunk_summaries.append(tokenizer_b.decode(outputs[0], skip_special_tokens=True))
            
        combined_summary = " ".join(chunk_summaries)
        final_inputs = tokenizer_b(combined_summary, return_tensors="pt", max_length=1024, truncation=True).to(device)
        with torch.no_grad():
            final_outputs = model_b.generate(final_inputs["input_ids"], max_length=150, num_beams=4, length_penalty=2.0)
        return tokenizer_b.decode(final_outputs[0], skip_special_tokens=True)

    print("--- CHẠY BARTPHO ---")
    for article in tqdm(articles, desc="BARTpho Inference"):
        normal_preds.append(normal_summarize(article))
        hierarchical_preds.append(hierarchical_sliding_summarize(article))
        
    del model_b
    torch.cuda.empty_cache()
else:
    print("Không tìm thấy model BARTpho.")

## 3. Qwen2.5: Native Long-Context

In [ ]:
QWEN_LORA_PATH = "./qwen-lora-vietnews"
QWEN_BASE = "Qwen/Qwen2.5-0.5B"
qwen_preds = []

if os.path.exists(QWEN_LORA_PATH):
    base_qwen = AutoModelForCausalLM.from_pretrained(QWEN_BASE, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
    model_qwen = PeftModel.from_pretrained(base_qwen, QWEN_LORA_PATH)
    model_qwen.eval()
    tokenizer_q = AutoTokenizer.from_pretrained(QWEN_BASE, trust_remote_code=True)
    if tokenizer_q.pad_token is None: tokenizer_q.pad_token = tokenizer_q.eos_token
    
    print("--- CHẠY QWEN ---")
    for article in tqdm(articles, desc="Qwen Long-Context Inference"):
        prompt = f"Tóm tắt bài báo sau:\n{article}\n\nTóm tắt:\n"
        # max_length=2048 để đọc toàn bộ article
        inputs = tokenizer_q(prompt, max_length=2048, truncation=True, return_tensors="pt").to(device)
        prompt_length = inputs["input_ids"].shape[1]
        
        with torch.no_grad():
            outputs = model_qwen.generate(**inputs, max_new_tokens=150, num_beams=4, pad_token_id=tokenizer_q.eos_token_id)
            
        summary_ids = outputs[0][prompt_length:]
        qwen_preds.append(tokenizer_q.decode(summary_ids, skip_special_tokens=True))
        
    del model_qwen
    torch.cuda.empty_cache()
else:
    print("Không tìm thấy model Qwen.")

## 4. Đánh giá So sánh ROUGE

In [ ]:
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
bertscore = evaluate.load("bertscore")

def evaluate_summaries(predictions, refs):
    r_score = rouge.compute(predictions=predictions, references=refs)
    bleu_score = bleu.compute(predictions=predictions, references=refs)
    bert_result = bertscore.compute(predictions=predictions, references=refs, lang="vi")
    
    return {
        "ROUGE-1": round(r_score['rouge1'] * 100, 2), 
        "ROUGE-2": round(r_score['rouge2'] * 100, 2),
        "ROUGE-L": round(r_score['rougeL'] * 100, 2),
        "ROUGE-Lsum": round(r_score['rougeLsum'] * 100, 2),
        "BLEU": round(bleu_score['bleu'] * 100, 2),
        "BERT-F1": round(np.mean(bert_result['f1']) * 100, 2)
    }

if normal_preds and hierarchical_preds:
    print("\n--- BARTPHO NORMAL (TRUNCATION AT 1024) ---")
    print(evaluate_summaries(normal_preds, references))
    print("\n--- BARTPHO HIERARCHICAL SLIDING WINDOW (MAP-REDUCE) ---")
    print(evaluate_summaries(hierarchical_preds, references))

if qwen_preds:
    print("\n--- QWEN NATIVE LONG-CONTEXT (2048 TOKENS) ---")
    print(evaluate_summaries(qwen_preds, references))

df_results = pd.DataFrame({
    "article": articles,
    "reference": references,
    "bartpho_normal": normal_preds if normal_preds else [None]*len(articles),
    "bartpho_hierarchical": hierarchical_preds if hierarchical_preds else [None]*len(articles),
    "qwen_long_context": qwen_preds if qwen_preds else [None]*len(articles)
})
df_results.to_csv("long_document_results.csv", index=False)
print("Đã lưu long_document_results.csv")

In [ ]:
# [POST-PROCESSING & RE-EVALUATION]
# Thêm bởi AI Reviewer để fix lỗi hallucination của Qwen Base Model
import pandas as pd
import evaluate

try:
    df_long = pd.read_csv("../long_document_results.csv")
except:
    df_long = pd.read_csv("long_document_results.csv")

def process_qwen(text):
    text = str(text)
    idx = text.find('\n\nTóm tắt:')
    if idx == -1: idx = text.find('\nTóm tắt:')
    if idx == -1: idx = text.find('Tóm tắt bài báo sau:')
    
    if idx != -1: text = text[:idx].strip()
        
    for token in ['<|endoftext|>', '<|im_end|>', '<|im_start|>']:
        text = text.replace(token, '')
    return text.strip()

print("Đang làm sạch output của Qwen2.5...")
df_long['qwen_lora'] = df_long['qwen_lora'].apply(process_qwen)

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

def quick_eval(predictions, references, name):
    print(f"\n--- Kết quả cho {name} ---")
    r_score = rouge.compute(predictions=predictions, references=references)
    print(f"ROUGE-1: {r_score['rouge1'] * 100:.2f}")
    print(f"ROUGE-2: {r_score['rouge2'] * 100:.2f}")
    print(f"ROUGE-L: {r_score['rougeL'] * 100:.2f}")
    print(f"ROUGE-Lsum: {r_score['rougeLsum'] * 100:.2f}")
    
    b_score = bleu.compute(predictions=predictions, references=[[r] for r in references])
    print(f"BLEU-4: {b_score['bleu'] * 100:.2f}")

refs_long = df_long['reference'].tolist()
quick_eval(df_long['qwen_lora'].tolist(), refs_long, "Qwen2.5 Long Context (Sau Post-Process)")
